# Exp 09: Full ratio weight

Card: [exp09_full_ratio_weight.md](../../docs/neuroslope/experiments/exp09_full_ratio_weight.md)


In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from training.config import ROOT
from training.config import VIP
from training.cv import CVResult
from training.runs import read_config
from analysis.gradcam import GradCamFigures
from training.tracking import start_run
from analysis.plots import pr_curves, save_figures, use_style

N_SHOW = 4
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp09_full_ratio_weight" / "resnet18.yaml"
BASELINE = CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp09_full_ratio_weight"

# dataviz reference palette, light mode; exp04 keeps its color from exp07
SERIES = {"damped (exp04)": "#eb6834", "full ratio": "#2a78d6"}
use_style()

## Training


In [ ]:
results = {"damped (exp04)": CVResult.load(BASELINE)}

In [ ]:
results["full ratio"] = CVResult.load(CONFIG)

## Training curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for key, ax in zip(["train_map", "val_map"], axes):
    for name, r in results.items():
        ax.plot(
            np.arange(1, len(r.mean_curve("val_map")) + 1),
            r.mean_curve(key),
            color=SERIES[name],
            lw=2,
            label=name,
        )
    ax.set(title=f"{key}, mean over folds", xlabel="epoch")
axes[1].legend(loc="lower right", title="pos_weight", alignment="left")
fig.tight_layout()
figures = {f"{PREFIX}_training.png": fig}
plt.show()

## AP and Macro F1


In [ ]:
scores = pd.concat(
    {name: r.fold_scores() for name, r in results.items()}, names=["pos_weight"]
)
scores.groupby("pos_weight", sort=False).mean().round(3)

In [ ]:
# counts only if same sign on both folds
diff = (
    results["full ratio"].fold_scores() - results["damped (exp04)"].fold_scores()
).T.add_prefix("fold ")
diff["same sign"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Precision-recall curves


In [ ]:
fig = pr_curves(results, VIP, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with start_run(read_config(CONFIG)["experiment"], "summary", {"val_mode": "fast"}):
    mlflow.log_table(scores.reset_index(), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")

save_figures(figures, FIGURES)